# Experiments

Anton Smirnov 27-29.03.2026

In [1]:
import os
import json
import pandas as pd
from glob import glob
from pathlib import Path

In [2]:
%load_ext slurm_magic

In [3]:
min_level=3
max_level=13 #not include 13
model_type = 'mna'
param_base_path = "/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/params"

In [4]:
#!rm -rf /projects/rat_gravitation/anton_disser/pass-nextflow-work/??

## Charged vs neutral

### Charged

In [5]:
base_path = "/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026"
train_files = sorted(glob(f"{base_path}/*.csv"))
print(train_files)

['/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_epitope_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_epitope_mouse.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_mhc_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_mhc_mouse.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_epitope_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_epitope_mouse.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_mhc_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_mhc_mouse.csv']


In [6]:
samplesheet = pd.DataFrame(columns = ["train_file","test_file","model_type","mna_level"])
i = 0
for tr in train_files:
    for lev in range(min_level,max_level):
        samplesheet.loc[i] = [str(Path(tr).resolve()),None,model_type, lev]
        i+=1
print(samplesheet.shape)
samplesheet.head()

(80, 4)


,train_file,test_file,model_type,mna_level
0,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,3
1,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,4
2,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,5
3,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,6
4,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,7


In [7]:
run_config = {
    "input": f"{param_base_path}/charged.csv",
    "output": "/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/charged",
    "structure_column": "Structure",
    "activity": "Activity",
    "run_name": "TCR-ch",    
    "publish_model": True,
    "charged": True,
    "validate": True,
    "filter_bad": True,
    "test_records_id_colname": "id",
    "separator_in_files": ";"
}

with open(f"{param_base_path}/charged.json","w") as file:
    file.write(json.dumps(run_config))
    
samplesheet.to_csv(f"{param_base_path}/charged.csv",index=False)

In [8]:
%%sbatch
#!/bin/bash
#SBATCH --job-name=TCR-ch    # Job name
#SBATCH --cpus-per-task=2        # Run on a single CPU
#SBATCH --mem=8gb                 # Job memory request
#SBATCH --time=5-23:59:00           # Time limit hrs:min:sec
#SBATCH --constraint=hpc
#SBATCH --output=/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/logs/TCR-ch.log   # Standard output and error log
#SBATCH --partition=long         # Task priority
#SBATCH --mail-type=ALL
#SBATCH --mail-user=SmirnygaTotoshka@yandex.ru

mkdir -p /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/charged
cd /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/charged
rm -rf /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/charged/pipeline_info

conda run -n nextflow \
nextflow run /home/asmirnov/dissertation_new/pass-nextflow/main.nf \
-profile aldan -with-report \
-w /projects/rat_gravitation/anton_disser/pass-nextflow-work/ \
-params-file /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/params/charged.json \
-queue-size 15 -resume

'Submitted batch job 1200689\n'

In [9]:
!tail -n 50 /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/logs/TCR-ch.log

| Процент успеха                                   | 48.8%        |
+--------------------------------------------------+---------------+
| 🧪 ТЕСТИРОВАНИЕ                                  |               |
| Уникальных тестирований                          | 0            |
| Успешно проведено                                | 0            |
| Процент успеха                                   | 0%           |
+--------------------------------------------------+---------------+

✓ Статистика сохранена: /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/charged/report.txt
[a3/1836b8] NOTE: Process `PASS:GET_RESULT_TABLE (25de1c0380988a73afeaedb82e04b091)` terminated with an error exit status (1) -- Error is ignored

executor >  slurm (169)
[d8/f0689f] PREPARE_DATA:CREATE_FILES_MAP  | 1 of 1, cached: 1 ✔
[cf/811c90] PRE…T (cdr3beta_mhc_human.csv) | 8 of 8 ✔
[f9/c0213a] PREPARE_DATA:SAVE_SDF_LIST     | 1 of 1 ✔
[5c/1bf783] PRE…ATA:MAP_CONVERTED_TO_TASKS | 1 of 1 ✔
[92/fd3b86] PAS…38

### Neutral

In [10]:
base_path = "/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026"
train_files = sorted(glob(f"{base_path}/*.csv"))
print(train_files)

['/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_epitope_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_epitope_mouse.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_mhc_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3alpha_mhc_mouse.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_epitope_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_epitope_mouse.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_mhc_human.csv', '/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/data/v2026/cdr3beta_mhc_mouse.csv']


In [11]:
samplesheet = pd.DataFrame(columns = ["train_file","test_file","model_type","mna_level"])
i = 0
for tr in train_files:
    for lev in range(min_level,max_level):
        samplesheet.loc[i] = [str(Path(tr).resolve()),None,model_type, lev]
        i+=1
print(samplesheet.shape)
samplesheet.head()

(80, 4)


,train_file,test_file,model_type,mna_level
0,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,3
1,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,4
2,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,5
3,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,6
4,/projects/rat_gravitation/anton_disser/TCR-Pre...,None,mna,7


In [12]:
run_config = {
    "input": f"{param_base_path}/neutral.csv",
    "output": "/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/neutral",
    "structure_column": "Structure",
    "activity": "Activity",
    "run_name": "TCR-n",    
    "publish_model": True,
    "charged": False,
    "validate": True,
    "filter_bad": True,
    "test_records_id_colname": "id",
    "separator_in_files": ";"
}

with open(f"{param_base_path}/neutral.json","w") as file:
    file.write(json.dumps(run_config))
    
samplesheet.to_csv(f"{param_base_path}/neutral.csv",index=False)

In [13]:
%%sbatch
#!/bin/bash
#SBATCH --job-name=TCR-n    # Job name
#SBATCH --cpus-per-task=2        # Run on a single CPU
#SBATCH --mem=8gb                 # Job memory request
#SBATCH --time=5-23:59:00           # Time limit hrs:min:sec
#SBATCH --constraint=hpc
#SBATCH --output=/projects/rat_gravitation/anton_disser/TCR-Pred/03_26/logs/TCR-n.log   # Standard output and error log
#SBATCH --partition=long         # Task priority
#SBATCH --mail-type=ALL
#SBATCH --mail-user=SmirnygaTotoshka@yandex.ru

mkdir -p /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/neutral
cd /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/neutral
rm -rf /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/neutral/pipeline_info

conda run -n nextflow \
nextflow run /home/asmirnov/dissertation_new/pass-nextflow/main.nf \
-profile aldan -with-report \
-w /projects/rat_gravitation/anton_disser/pass-nextflow-work/ \
-params-file /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/params/neutral.json \
-queue-size 15 -resume

'Submitted batch job 1200690\n'

In [14]:
!tail -n 50 /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/logs/TCR-n.log

+--------------------------------------------------+---------------+
| 🧪 ТЕСТИРОВАНИЕ                                  |               |
| Уникальных тестирований                          | 0            |
| Успешно проведено                                | 0            |
| Процент успеха                                   | 0%           |
+--------------------------------------------------+---------------+

✓ Статистика сохранена: /projects/rat_gravitation/anton_disser/TCR-Pred/03_26/results/neutral/report.txt
[2e/856419] NOTE: Process `PASS:GET_RESULT_TABLE (25de1c0380988a73afeaedb82e04b091)` terminated with an error exit status (1) -- Error is ignored
[d9/0daae2] NOTE: Process `PASS:GET_RESULT_TABLE (4ddda368a718af62f28da659a1017e55)` terminated with an error exit status (1) -- Error is ignored

executor >  slurm (171)
[45/3416a6] PREPARE_DATA:CREATE_FILES_MAP  | 1 of 1 ✔
[c9/f23453] PRE…T (cdr3beta_mhc_human.csv) | 8 of 8 ✔
[21/3d2fde] PREPARE_DATA:SAVE_SDF_LIST     | 1 of 1 ✔
[eb/d

## Old vs new